# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shoaib237124/FlyRank_Internship_ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os
import duckdb
from google.colab import userdata

# 1. Retrieve your secret key from Google Colab Secrets
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Could not retrieve 'HF_TOKEN' from Colab Secrets. Ensure the key name matches exactly and notebook access is toggled ON in the Secrets tab.") from e

# 2. Verify token was found before passing to DuckDB
if not HF_TOKEN:
    raise ValueError("HF_TOKEN retrieved from secrets is empty.")

# 3. Connect to DuckDB and create secret
con = duckdb.connect()

# DuckDB >= 0.9 native Hugging Face secret syntax
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}');")

print("DuckDB Hugging Face secret successfully created!")

DuckDB Hugging Face secret successfully created!


In [6]:
import os
import duckdb
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

# 1. My Rule and its Reason Codes

## Baseline Rule

My baseline ranks pages for refresh based on three historical signals:

1. The page has high search visibility (many Google Search impressions).
2. The page has a low click-through rate relative to its visibility.
3. The page ranks outside the top search positions (higher average position number).

Pages satisfying these conditions receive higher priority because improving their titles, meta descriptions, or content may increase organic traffic.

This rule uses only historical information available at the decision time and does not use future information, labels, or FlyRank product flags.

## Reason Codes

The rule can assign one of the following reason codes:

- LOW_CTR_HIGH_IMPRESSIONS
- HIGH_IMPRESSIONS_POOR_POSITION
- LOW_VISIBILITY
- GOOD_PERFORMER

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
query = f"""
WITH page_stats AS (

SELECT

content_hash_id,

SUM(gsc_impressions) AS impressions,

SUM(gsc_clicks) AS clicks,

AVG(gsc_avg_position) AS avg_position,

SUM(ga4_sessions) AS sessions,

CASE
WHEN SUM(gsc_impressions)=0 THEN 0
ELSE 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
END AS ctr

FROM {TABLES['fact_daily']}

WHERE

month='2026-03'

AND gsc_data_available IS TRUE

AND ga4_data_available IS TRUE

GROUP BY content_hash_id

)

SELECT *

FROM page_stats

LIMIT 10;
"""

page_stats = con.sql(query).df()

page_stats


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,impressions,clicks,avg_position,sessions,ctr
0,content_5c80451459c29b4a,5.0,0.0,5.400000,1.0,0.000000
1,content_6b0149a80607dac3,1199.0,12.0,8.119012,26.0,1.000834
2,content_62673eea26c31c17,57145.0,43.0,6.814939,125.0,0.075247
3,content_872342e050545a12,39.0,0.0,6.538462,1.0,0.000000
4,content_4c185d1c173cd53d,278.0,9.0,10.399210,16.0,3.237410
5,content_bd07be40ea0d5f54,242.0,0.0,24.231436,18.0,0.000000
6,content_40e28f4b41764012,497.0,6.0,5.972594,16.0,1.207243
7,content_f8b204c8ce80dad5,28.0,1.0,7.219841,8.0,3.571429
8,content_c943a83124c43e95,1.0,0.0,5.000000,1.0,0.000000
9,content_a0e19c582cf792a7,143.0,3.0,5.887813,4.0,2.097902


In [8]:
# Signal 1  CTR vs Average Position
query = f"""
WITH page_stats AS (

SELECT

content_hash_id,

SUM(gsc_impressions) AS impressions,

SUM(gsc_clicks) AS clicks,

AVG(gsc_avg_position) AS avg_position,

CASE
WHEN SUM(gsc_impressions)=0 THEN 0
ELSE 100.0 * SUM(gsc_clicks)/SUM(gsc_impressions)
END AS ctr

FROM {TABLES['fact_daily']}

WHERE

month='2026-03'

AND gsc_data_available IS TRUE

GROUP BY content_hash_id

)

SELECT

CASE

WHEN avg_position <=5 THEN '1-5'

WHEN avg_position<=10 THEN '6-10'

WHEN avg_position<=20 THEN '11-20'

ELSE '20+'

END AS position_bucket,

COUNT(*) AS n,

ROUND(AVG(ctr),2) AS avg_ctr

FROM page_stats

GROUP BY position_bucket

ORDER BY position_bucket;
"""

signal1 = con.sql(query).df()

signal1

,position_bucket,n,avg_ctr
0,1-5,44171,0.88
1,11-20,32203,0.32
2,20+,44969,0.19
3,6-10,55395,0.42


## Signal Check 1 – CTR vs Average Position

**Signal:** CTR compared with average Google Search position.

| Position Bucket | n | Average CTR (%) |
|----------------|---------:|--------------:|
| 1–5 | 44,171 | 0.88 |
| 6–10 | 55,395 | 0.42 |
| 11–20 | 32,203 | 0.32 |
| 20+ | 44,969 | 0.19 |

### Verdict: **CONFIRMED**

Pages ranking higher in Google Search receive higher average CTR, while pages with worse average positions receive substantially lower CTR.

This confirms that **CTR opportunity combined with search position** is a useful signal for prioritizing content refreshes. Pages with many impressions but low CTR may benefit from improving titles, metadata, or content quality.

In [9]:
#Signal Check 2 (Search Visibility)
query = f"""
WITH page_stats AS (

SELECT

content_hash_id,

SUM(gsc_impressions) AS impressions

FROM {TABLES['fact_daily']}

WHERE

month='2026-03'

AND gsc_data_available IS TRUE

GROUP BY content_hash_id

)

SELECT

CASE

WHEN impressions<100 THEN '<100'

WHEN impressions<500 THEN '100-499'

WHEN impressions<1000 THEN '500-999'

WHEN impressions<5000 THEN '1000-4999'

ELSE '5000+'

END AS impression_bucket,

COUNT(*) AS n

FROM page_stats

GROUP BY impression_bucket

ORDER BY impression_bucket;
"""

signal2 = con.sql(query).df()

signal2

,impression_bucket,n
0,100-499,39517
1,1000-4999,31766
2,500-999,16866
3,5000+,13292
4,<100,75297


## Signal Check 2 – Search Visibility

**Signal:** Distribution of search impressions.

| Impression Bucket | n |
|------------------|------:|
| <100 | 75,297 |
| 100–499 | 39,517 |
| 500–999 | 16,866 |
| 1000–4999 | 31,766 |
| 5000+ | 13,292 |

### Verdict: **CONFIRMED**

Most pages receive fewer than 100 impressions, while a much smaller group receives thousands of impressions.

This supports using impressions as part of the baseline rule because pages with higher visibility offer larger potential gains from content improvements.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

query = f"""
WITH page_stats AS (

SELECT

content_hash_id,
client_hash_id,

SUM(gsc_impressions) AS impressions,
SUM(gsc_clicks) AS clicks,

AVG(gsc_avg_position) AS avg_position,

SUM(ga4_sessions) AS sessions,

CASE
WHEN SUM(gsc_impressions)=0 THEN 0
ELSE 100.0*SUM(gsc_clicks)/SUM(gsc_impressions)
END AS ctr

FROM {TABLES['fact_daily']}

WHERE

month='2026-03'

AND gsc_data_available IS TRUE
AND ga4_data_available IS TRUE

GROUP BY
content_hash_id,
client_hash_id

)

SELECT *

FROM page_stats;
"""

df = con.sql(query).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [11]:
#SCORE
df["score"] = 0

# High impressions
df.loc[df["impressions"] >= 1000, "score"] += 40

# Low CTR
df.loc[df["ctr"] < 1.0, "score"] += 30

# Poor ranking
df.loc[df["avg_position"] > 10, "score"] += 30

In [12]:
# Reason Codes
def reason(row):

    if row.impressions >= 1000 and row.ctr < 1:
        return "LOW_CTR_HIGH_IMPRESSIONS"

    elif row.avg_position > 10:
        return "HIGH_IMPRESSIONS_POOR_POSITION"

    elif row.impressions < 100:
        return "LOW_VISIBILITY"

    else:
        return "GOOD_PERFORMER"


df["reason_code"] = df.apply(reason, axis=1)

In [13]:
# Action labels
def action(score):

    if score >= 70:
        return "REFRESH_NOW"

    elif score >= 40:
        return "REVIEW"

    else:
        return "MONITOR"


df["action"] = df["score"].apply(action)

In [14]:
# RANK
df = df.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

df["rank"] = df.index + 1

In [15]:
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"

df.to_csv(output_path, index=False)

print(output_path)

df.head(20)

work/outputs/baseline_action_score.csv


,content_hash_id,client_hash_id,impressions,clicks,avg_position,sessions,ctr,score,reason_code,action,rank
0,content_d9141abc29f56e8b,client_1a730cb2640a1abf,6751.0,17.0,10.541426,45.0,0.251815,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,1
1,content_917d2b8a921cc916,client_23a62021009f63c4,2114.0,8.0,16.295610,32.0,0.378430,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,2
2,content_80598853ffeb32e3,client_23a62021009f63c4,1029.0,3.0,29.495455,38.0,0.291545,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,3
3,content_18cfb571158fa8cb,client_23a62021009f63c4,1940.0,4.0,38.933388,26.0,0.206186,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,4
4,content_65407c073e92b89c,client_23a62021009f63c4,2917.0,2.0,40.436665,77.0,0.068564,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,5
5,content_31932a8cdcccd397,client_e547b89c05043229,2245.0,14.0,11.070310,16.0,0.623608,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,6
6,content_86cdde17fc8a8f7f,client_fef1a8f436438636,1606.0,3.0,24.109025,28.0,0.186800,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,7
7,content_40cb744b7c2b0dee,client_fef1a8f436438636,1175.0,11.0,16.400929,16.0,0.936170,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,8
8,content_104ac5260b33526e,client_1a730cb2640a1abf,1002.0,6.0,11.734009,11.0,0.598802,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,9
9,content_eedbbce7a7ff552d,client_23a62021009f63c4,1103.0,0.0,30.619323,40.0,0.000000,100,LOW_CTR_HIGH_IMPRESSIONS,REFRESH_NOW,10


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

# 3. Top-20 Review

The highest-ranked pages all received the **REFRESH_NOW** action because they satisfy the baseline rule:

- High search visibility (high impressions)
- Low click-through rate (CTR below 1%)
- Poor average search position (greater than 10)

These pages represent the highest-priority refresh opportunities according to the baseline.

| Rank | Action | Reason Code | Confidence | What would make it wrong? |
|------|---------|-------------|------------|----------------------------|
| 1 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | High | The page was recently updated but the warehouse has not yet reflected the improvement. |
| 2 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | High | The topic is highly seasonal, making March performance unrepresentative. |
| 3 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | High | Search intent has changed and a refresh alone will not improve rankings. |
| 4 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | High | Competitors dominate the search results for reasons unrelated to content quality. |
| 5 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | High | The page targets outdated or declining search demand. |
| 6 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | High | The page already has an approved refresh planned. |
| 7 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | The page may be technically limited (indexing or structured data issues). |
| 8 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | CTR may improve through title changes rather than a full content refresh. |
| 9 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | Search intent may no longer match the page's purpose. |
| 10 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | Traffic may fluctuate because of temporary ranking volatility. |
| 11 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | Zero clicks could result from a temporary reporting issue rather than poor performance. |
| 12 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | Engagement may improve without requiring a full refresh. |
| 13 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | The page may already be optimized for a niche query. |
| 14 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | External events may have temporarily reduced demand. |
| 15 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | The page may primarily support branding rather than traffic. |
| 16 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | A low CTR may result from rich SERP features rather than poor content. |
| 17 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | Search competition may be unusually strong during this period. |
| 18 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | The page may require technical SEO fixes instead of content changes. |
| 19 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | The page may target a very specific audience with naturally low CTR. |
| 20 | REFRESH_NOW | LOW_CTR_HIGH_IMPRESSIONS | Medium | The page may perform differently over a longer observation window. |

---



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

# 4. Weak Picks + Leakage Check

## Weak Picks

Although the baseline is transparent, it has several limitations:

- It assumes every page with high impressions and low CTR should be refreshed, which may not always be true.
- Some pages may have low CTR because of search intent, strong competitors, or SERP features rather than poor content.
- Seasonal pages may appear as refresh opportunities even though their traffic naturally varies during the year.
- The baseline uses fixed thresholds and does not learn complex relationships between impressions, CTR, rankings, and user engagement.

These limitations are expected because this is a rule-based baseline. A machine learning model should improve on this by learning patterns from multiple signals simultaneously.

---

## Leakage Check

The baseline was checked for information leakage.

**Included features**

- Historical Google Search impressions
- Historical Google Search clicks
- Historical average search position
- Historical GA4 sessions

**Excluded**

- Future observations
- Future performance windows
- Label-derived columns
- FlyRank product flags
- Client and content identifiers as predictive features

Therefore, the baseline uses only information that would have been available at the time the refresh decision was made.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.